# Incident Response Runbook: Execution - User Execution

**Tactic:** Execution
**Technique:** User Execution (T1204)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for User Execution activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
user_execution_indicators = []
incident_id = None

# Query Splunk for user execution indicators
print(f"\n[DETECTION] Querying Splunk for user execution activities...")
try:
    # Malicious file execution by users
    file_exec_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688
    | eval risk_score = case(
        match(Image, "\.exe$|\.scr$|\.pif$|\.com$") AND match(CommandLine, "temp|downloads|desktop"), 8,
        match(Image, "\.hta$|\.vbs$|\.js$|\.jse$|\.vbe$"), 9,
        match(Image, "\.bat$|\.cmd$") AND match(CommandLine, "suspicious|encoded"), 7,
        match(Image, "\.msi$|\.msp$") AND match(CommandLine, "/quiet|/passive"), 6,
        match(CommandLine, "rundll32\.exe.*javascript:|regsvr32.*scrobj"), 10,
        1==1, 3
    )
    | where risk_score >= 6
    | stats count by host, user, Image, CommandLine, risk_score
    | sort -risk_score
    """

    file_results = splunk.search(file_exec_query, timeframe="-24h")
    if file_results:
        print(f"   Found {len(file_results)} suspicious file executions")
        for result in file_results[:10]:  # Top 10
            user_execution_indicators.append({
                'type': 'malicious_file_execution',
                'host': result.get('host'),
                'user': result.get('user'),
                'image': result.get('Image'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Suspicious process execution patterns
    process_exec_query = """
    index=* sourcetype=WinEventLog:Security EventCode=4688
    | eval risk_score = case(
        match(CommandLine, "powershell.*-executionpolicy.*bypass"), 10,
        match(CommandLine, "cmd\.exe.*/c.*echo.*\|"), 9,
        match(CommandLine, "rundll32\.exe.*javascript:"), 10,
        match(CommandLine, "regsvr32.*scrobj.*dll"), 9,
        match(CommandLine, "mshta.*vbscript:"), 8,
        match(CommandLine, "wscript.*cscript.*encoded"), 8,
        1==1, 4
    )
    | where risk_score >= 7
    | stats count by host, user, CommandLine, risk_score
    | sort -risk_score
    """

    process_results = splunk.search(process_exec_query, timeframe="-24h")
    if process_results:
        print(f"   Found {len(process_results)} suspicious process executions")
        for result in process_results[:10]:  # Top 10
            user_execution_indicators.append({
                'type': 'suspicious_process_execution',
                'host': result.get('host'),
                'user': result.get('user'),
                'command': result.get('CommandLine'),
                'risk_score': result.get('risk_score')
            })

    # Email attachment execution
    email_attach_query = """
    index=* sourcetype=email sourcetype=exchange
    | search "attachment.*opened" OR "file.*executed" OR "double-clicked"
    | eval risk_score = case(
        match(attachment_name, "\.exe$|\.scr$|\.pif$|\.hta$|\.vbs$"), 9,
        match(attachment_name, "\.zip$|\.rar$") AND match(subject, "urgent|important|invoice"), 8,
        match(attachment_name, "\.docm$|\.xlsm$|\.pptm$"), 7,
        match(subject, "password|credential|account|login"), 6,
        1==1, 4
    )
    | where risk_score >= 6
    | stats count by sender, recipient, subject, attachment_name, risk_score
    | sort -risk_score
    """

    email_results = splunk.search(email_attach_query, timeframe="-24h")
    if email_results:
        print(f"   Found {len(email_results)} suspicious email attachment executions")
        for result in email_results[:10]:  # Top 10
            user_execution_indicators.append({
                'type': 'email_attachment_execution',
                'sender': result.get('sender'),
                'recipient': result.get('recipient'),
                'subject': result.get('subject'),
                'attachment': result.get('attachment_name'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Get unique affected systems/users
unique_hosts = set()
unique_users = set()
for indicator in user_execution_indicators:
    if indicator.get('host'):
        unique_hosts.add(indicator['host'])
    if indicator.get('recipient') or indicator.get('user'):
        unique_users.add(indicator.get('recipient') or indicator.get('user'))

# Check CrowdStrike for endpoint details
print(f"\n[DETECTION] Checking CrowdStrike for endpoint details...")
for host in unique_hosts:
    try:
        cs_info = crowdstrike.get_host_info(host)
        if cs_info:
            affected_systems.append({
                'hostname': host,
                'device_id': cs_info.get('device_id'),
                'os': cs_info.get('os_version'),
                'last_seen': cs_info.get('last_seen'),
                'activities': [a for a in user_execution_indicators if a.get('host') == host]
            })
            print(f"   Found endpoint details for: {host}")
    except Exception as e:
        print(f"   CrowdStrike check failed for {host}: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
for indicator in user_execution_indicators[:5]:  # Check top 5
    try:
        if indicator.get('attachment'):
            ti_data = misp.lookup_filename(indicator['attachment'])
            if ti_data:
                indicator['threat_intel'] = ti_data
                print(f"   Threat intel found for attachment: {indicator['attachment']}")
        elif indicator.get('command'):
            # Extract file names from commands
            file_names = re.findall(r'[\w\-\.]+\.(exe|vbs|js|hta|bat|cmd|scr|pif)', indicator['command'], re.IGNORECASE)
            for file_name in file_names:
                ti_data = misp.lookup_filename(file_name)
                if ti_data:
                    indicator['threat_intel'] = ti_data
                    print(f"   Threat intel found for file: {file_name}")
                    break
    except Exception as e:
        print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'User Execution Activities - {len(user_execution_indicators)} indicators',
        'description': f'Detected suspicious user execution activities affecting {len(unique_users)} users across {len(affected_systems)} systems',
        'severity': 'HIGH',
        'tactic': 'Execution',
        'technique': 'User Execution (T1204)',
        'indicators': user_execution_indicators[:10],  # Top 10 indicators
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(user_execution_indicators)} user execution indicators detected")
print(f"  - {len(unique_users)} affected users identified")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - Threat intelligence enriched for {len([i for i in user_execution_indicators if i.get('threat_intel')])} indicators")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []
containment_time = datetime.now().isoformat()

# 1. Isolate affected systems via CrowdStrike
print(f"\n[CONTAINMENT] Isolating affected systems...")
isolated_hosts = []
for system in affected_systems:
    try:
        result = crowdstrike.isolate_host(system['device_id'])
        if result:
            isolated_hosts.append(system['hostname'])
            containment_actions.append({
                'action': 'host_isolation',
                'target': system['hostname'],
                'device_id': system['device_id'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Isolated host: {system['hostname']}")
        else:
            print(f"   Failed to isolate host: {system['hostname']}")
    except Exception as e:
        print(f"   Error isolating {system['hostname']}: {e}")

# 2. Block suspicious IPs/domains via Shuffle
print(f"\n[CONTAINMENT] Blocking suspicious IPs/domains...")
blocked_indicators = []
for indicator in user_execution_indicators:
    try:
        # Extract IPs from commands or logs
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        ips = re.findall(ip_pattern, str(indicator))
        for ip in ips:
            if ip not in ['127.0.0.1', '0.0.0.0']:  # Skip localhost
                result = shuffle.block_ip(ip, f"User execution incident {incident_id}")
                if result:
                    blocked_indicators.append({'type': 'ip', 'value': ip})
                    containment_actions.append({
                        'action': 'ip_block',
                        'target': ip,
                        'reason': f'User execution from {indicator.get("host", "unknown")}',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked IP: {ip}")

        # Extract domains from commands
        domain_pattern = r'\b([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}\b'
        domains = re.findall(domain_pattern, str(indicator))
        for domain in domains:
            if not any(skip in domain.lower() for skip in ['localhost', 'localdomain']):
                result = shuffle.block_domain(domain, f"User execution incident {incident_id}")
                if result:
                    blocked_indicators.append({'type': 'domain', 'value': domain})
                    containment_actions.append({
                        'action': 'domain_block',
                        'target': domain,
                        'reason': f'User execution from {indicator.get("host", "unknown")}',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked domain: {domain}")
    except Exception as e:
        print(f"   Error blocking indicators: {e}")

# 3. Disable compromised user accounts
print(f"\n[CONTAINMENT] Disabling compromised user accounts...")
disabled_accounts = []
for user in unique_users:
    try:
        # Use Shuffle to disable accounts (assuming AD integration)
        result = shuffle.disable_user_account(user, f"User execution incident {incident_id}")
        if result:
            disabled_accounts.append(user)
            containment_actions.append({
                'action': 'account_disable',
                'target': user,
                'reason': 'Compromised account - user execution activities',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Disabled account: {user}")
        else:
            print(f"   Failed to disable account: {user}")
    except Exception as e:
        print(f"   Error disabling account {user}: {e}")

# 4. Enable enhanced monitoring
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    # Create Splunk alerts for enhanced monitoring
    alert_configs = [
        {
            'name': f'Enhanced User Execution Monitoring - {incident_id}',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "powershell.*-executionpolicy.*bypass"), 10, match(CommandLine, "cmd\.exe.*/c.*echo.*\|"), 9, 1==1, 4) | where risk_score >= 7',
            'alert_type': 'real-time',
            'severity': 'high'
        }
    ]

    for config in alert_configs:
        result = splunk.create_alert(config)
        if result:
            containment_actions.append({
                'action': 'enhanced_monitoring',
                'target': 'splunk_alert',
                'alert_name': config['name'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Created enhanced monitoring alert: {config['name']}")

    # Enable CrowdStrike enhanced detection
    for system in affected_systems:
        result = crowdstrike.enable_enhanced_detection(system['device_id'])
        if result:
            containment_actions.append({
                'action': 'enhanced_edr',
                'target': system['hostname'],
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Enabled enhanced EDR for: {system['hostname']}")

except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

# Update IRIS case with containment actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'containment_actions': containment_actions,
            'containment_time': containment_time,
            'isolated_hosts': isolated_hosts,
            'blocked_indicators': blocked_indicators,
            'disabled_accounts': disabled_accounts
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with containment details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} systems isolated")
print(f"  - {len(blocked_indicators)} IPs/domains blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - Enhanced monitoring enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []
eradication_time = datetime.now().isoformat()

# 1. Terminate malicious processes via CrowdStrike
print(f"\n[ERADICATION] Terminating malicious processes...")
terminated_processes = []
for system in affected_systems:
    try:
        # Get running processes from CrowdStrike
        processes = crowdstrike.get_processes(system['device_id'])
        if processes:
            malicious_processes = []
            for proc in processes:
                proc_name = proc.get('name', '').lower()
                proc_cmd = proc.get('command_line', '').lower()

                # Check against known malicious patterns
                if any(pattern in proc_cmd for pattern in [
                    'powershell -executionpolicy bypass',
                    'cmd.exe /c echo',
                    'rundll32 javascript:',
                    'regsvr32 scrobj.dll',
                    'mshta vbscript:',
                    'wscript encoded',
                    'cscript encoded'
                ]) or any(pattern in proc_name for pattern in [
                    'powershell.exe', 'cmd.exe', 'wscript.exe', 'cscript.exe',
                    'mshta.exe', 'regsvr32.exe', 'rundll32.exe'
                ]):
                    malicious_processes.append(proc)

            # Terminate malicious processes
            for proc in malicious_processes[:5]:  # Limit to top 5 per host
                result = crowdstrike.terminate_process(system['device_id'], proc['pid'])
                if result:
                    terminated_processes.append({
                        'host': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'command': proc.get('command_line', '')
                    })
                    eradication_actions.append({
                        'action': 'process_termination',
                        'target': system['hostname'],
                        'process': proc['name'],
                        'pid': proc['pid'],
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated process: {proc['name']} (PID: {proc['pid']}) on {system['hostname']}")
    except Exception as e:
        print(f"   Error terminating processes on {system['hostname']}: {e}")

# 2. Delete malicious files
print(f"\n[ERADICATION] Deleting malicious files...")
deleted_files = []
for indicator in user_execution_indicators:
    try:
        if indicator.get('image') and indicator['image'].endswith(('.exe', '.scr', '.pif', '.hta', '.vbs', '.js', '.bat', '.cmd')):
            # Use CrowdStrike to delete file
            result = crowdstrike.delete_file(indicator['host'], indicator['image'])
            if result:
                deleted_files.append({
                    'host': indicator['host'],
                    'file': indicator['image']
                })
                eradication_actions.append({
                    'action': 'file_deletion',
                    'target': indicator['host'],
                    'file': indicator['image'],
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Deleted file: {indicator['image']} on {indicator['host']}")

        # Delete from temp directories
        temp_paths = [
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\AppData\\Local\\Temp\\*",
            f"C:\\Users\\{indicator.get('user', 'unknown')}\\Downloads\\*",
            "C:\\Windows\\Temp\\*"
        ]
        for temp_path in temp_paths:
            result = crowdstrike.delete_file(indicator['host'], temp_path)
            if result:
                print(f"   Cleaned temp directory: {temp_path} on {indicator['host']}")
    except Exception as e:
        print(f"   Error deleting files: {e}")

# 3. Reset compromised user passwords
print(f"\n[ERADICATION] Resetting compromised user passwords...")
reset_accounts = []
for user in unique_users:
    try:
        # Use Shuffle to reset passwords
        result = shuffle.reset_user_password(user, f"Compromised account - user execution incident {incident_id}")
        if result:
            reset_accounts.append(user)
            eradication_actions.append({
                'action': 'password_reset',
                'target': user,
                'reason': 'Compromised during user execution incident',
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Reset password for: {user}")
        else:
            print(f"   Failed to reset password for: {user}")
    except Exception as e:
        print(f"   Error resetting password for {user}: {e}")

# 4. Clean up persistence mechanisms
print(f"\n[ERADICATION] Cleaning up persistence mechanisms...")
cleaned_persistence = []
for system in affected_systems:
    try:
        # Check for common persistence locations
        persistence_checks = [
            'HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run',
            'HKLM\\Software\\Microsoft\\Windows\\CurrentVersion\\Run',
            'HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\RunOnce',
            'C:\\Users\\*\\AppData\\Roaming\\Microsoft\\Windows\\Start Menu\\Programs\\Startup\\*',
            'C:\\ProgramData\\Microsoft\\Windows\\Start Menu\\Programs\\StartUp\\*'
        ]

        for reg_key in persistence_checks:
            result = crowdstrike.clean_registry_key(system['device_id'], reg_key)
            if result:
                cleaned_persistence.append({
                    'host': system['hostname'],
                    'registry_key': reg_key
                })
                eradication_actions.append({
                    'action': 'registry_cleanup',
                    'target': system['hostname'],
                    'registry_key': reg_key,
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleaned registry: {reg_key} on {system['hostname']}")
    except Exception as e:
        print(f"   Error cleaning persistence on {system['hostname']}: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
verification_results = []
try:
    # Re-scan systems for remaining threats
    for system in affected_systems:
        scan_result = crowdstrike.scan_host(system['device_id'])
        if scan_result:
            verification_results.append({
                'host': system['hostname'],
                'scan_result': scan_result,
                'threats_found': len(scan_result.get('threats', []))
            })
            if scan_result.get('threats'):
                print(f"   Verification found {len(scan_result['threats'])} remaining threats on {system['hostname']}")
            else:
                print(f"   Verification clean: {system['hostname']}")
        else:
            print(f"   Verification scan failed for: {system['hostname']}")
except Exception as e:
    print(f"   Error during verification: {e}")

# Update IRIS case with eradication actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'eradication_actions': eradication_actions,
            'eradication_time': eradication_time,
            'terminated_processes': terminated_processes,
            'deleted_files': deleted_files,
            'reset_accounts': reset_accounts,
            'cleaned_persistence': cleaned_persistence,
            'verification_results': verification_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with eradication details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(deleted_files)} malicious files deleted")
print(f"  - {len(reset_accounts)} user passwords reset")
print(f"  - {len(cleaned_persistence)} persistence mechanisms cleaned")
print(f"  - Verification completed for {len(verification_results)} systems")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []
recovery_time = datetime.now().isoformat()

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
reenabled_hosts = []
for system in affected_systems:
    if system['hostname'] in isolated_hosts:
        try:
            result = crowdstrike.unisolate_host(system['device_id'])
            if result:
                reenabled_hosts.append(system['hostname'])
                recovery_actions.append({
                    'action': 'host_unisolation',
                    'target': system['hostname'],
                    'device_id': system['device_id'],
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {system['hostname']}")
            else:
                print(f"   Failed to re-enable host: {system['hostname']}")
        except Exception as e:
            print(f"   Error re-enabling {system['hostname']}: {e}")

# 2. Re-enable user accounts
print(f"\n[RECOVERY] Re-enabling user accounts...")
reenabled_accounts = []
for user in disabled_accounts:
    try:
        # Use Shuffle to re-enable accounts
        result = shuffle.enable_user_account(user, f"Recovery complete - user execution incident {incident_id}")
        if result:
            reenabled_accounts.append(user)
            recovery_actions.append({
                'action': 'account_enable',
                'target': user,
                'reason': 'Recovery complete - user execution incident resolved',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Re-enabled account: {user}")
        else:
            print(f"   Failed to re-enable account: {user}")
    except Exception as e:
        print(f"   Error re-enabling account {user}: {e}")

# 3. Restore normal operations
print(f"\n[RECOVERY] Restoring normal operations...")
restored_services = []
try:
    # Restore blocked IPs/domains (selective restoration)
    for indicator in blocked_indicators[:5]:  # Restore top 5 to avoid reinfection
        if indicator['type'] == 'ip':
            result = shuffle.unblock_ip(indicator['value'], f"Recovery complete - incident {incident_id}")
            if result:
                restored_services.append(f"IP: {indicator['value']}")
                recovery_actions.append({
                    'action': 'ip_unblock',
                    'target': indicator['value'],
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Unblocked IP: {indicator['value']}")

    # Restore normal monitoring levels
    result = splunk.disable_alert(f'Enhanced User Execution Monitoring - {incident_id}')
    if result:
        restored_services.append('Enhanced monitoring disabled')
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'splunk_alerts',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring levels")

except Exception as e:
    print(f"   Error restoring operations: {e}")

# 4. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
validation_results = []
for system in affected_systems:
    try:
        # Check system health via CrowdStrike
        health_check = crowdstrike.check_system_health(system['device_id'])
        if health_check:
            validation_results.append({
                'host': system['hostname'],
                'health_status': health_check.get('status', 'unknown'),
                'services_running': health_check.get('services', []),
                'connectivity': health_check.get('network_status', 'unknown')
            })

            if health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'health_validation',
                    'target': system['hostname'],
                    'status': 'success',
                    'health_status': 'healthy',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['hostname']}")
            else:
                print(f"   System health issues detected: {system['hostname']} - {health_check.get('issues', [])}")
        else:
            print(f"   Health check failed for: {system['hostname']}")
    except Exception as e:
        print(f"   Error validating {system['hostname']}: {e}")

# 5. Monitor for residual issues
print(f"\n[RECOVERY] Establishing post-recovery monitoring...")
try:
    # Create follow-up monitoring alerts
    followup_alerts = [
        {
            'name': f'Post-Recovery User Execution Monitoring - {incident_id}',
            'search': f'index=* host IN ({",".join([f'"{s["hostname"]}"' for s in affected_systems])}) sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "powershell.*-executionpolicy.*bypass"), 10, match(CommandLine, "cmd\.exe.*/c.*echo.*\|"), 9, 1==1, 4) | where risk_score >= 7',
            'alert_type': 'scheduled',
            'severity': 'medium',
            'schedule': '0 */4 * * *'  # Every 4 hours
        }
    ]

    for alert_config in followup_alerts:
        result = splunk.create_alert(alert_config)
        if result:
            recovery_actions.append({
                'action': 'followup_monitoring',
                'target': 'splunk_alert',
                'alert_name': alert_config['name'],
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Created follow-up monitoring: {alert_config['name']}")

except Exception as e:
    print(f"   Error establishing follow-up monitoring: {e}")

# Update IRIS case with recovery actions
if incident_id:
    try:
        iris.update_case(incident_id, {
            'recovery_actions': recovery_actions,
            'recovery_time': recovery_time,
            'reenabled_hosts': reenabled_hosts,
            'reenabled_accounts': reenabled_accounts,
            'restored_services': restored_services,
            'validation_results': validation_results
        })
        print(f"\n[CASE] Updated IRIS case {incident_id} with recovery details")
    except Exception as e:
        print(f"   Failed to update IRIS case: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(reenabled_hosts)} systems re-enabled")
print(f"  - {len(reenabled_accounts)} accounts re-enabled")
print(f"  - {len(restored_services)} services restored")
print(f"  - System validation completed for {len(validation_results)} systems")
print(f"  - Post-recovery monitoring established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'User Execution Incident Report - {len(user_execution_indicators)} indicators',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'HIGH',
        'tactic': 'Execution',
        'technique': 'User Execution (T1204)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'indicators_detected': len(user_execution_indicators),
            'hosts_isolated': len(isolated_hosts),
            'accounts_disabled': len(disabled_accounts),
            'processes_terminated': len(terminated_processes),
            'files_deleted': len(deleted_files),
            'accounts_reset': len(reset_accounts)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': user_execution_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in user_execution_indicators if i.get('threat_intel')],
        'recommendations': [
            'Implement user execution prevention controls',
            'Enhance email attachment scanning',
            'Deploy application whitelisting',
            'Conduct user security awareness training',
            'Review and update execution policies'
        ]
    }

    # Save report to file
    report_filename = f"incident_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection through Splunk correlation rules',
            'Effective containment via CrowdStrike isolation',
            'Comprehensive eradication of malicious processes',
            'Successful recovery and system validation'
        ],
        'what_could_improve': [
            'Earlier detection of user execution patterns',
            'Automated user training triggers',
            'Enhanced threat intelligence integration',
            'Proactive prevention measures'
        ],
        'key_findings': [
            f'User execution affected {len(unique_users)} users across {len(affected_systems)} systems',
            f'Most common vector: {max([i.get("type", "unknown") for i in user_execution_indicators], key=[i.get("type", "unknown") for i in user_execution_indicators].count)}',
            'Threat intelligence enriched detection for key indicators',
            'Automated response prevented lateral movement'
        ],
        'preventive_measures': [
            'Deploy application execution controls',
            'Implement user behavior analytics',
            'Enhance email security gateways',
            'Regular security awareness training',
            'Update endpoint protection policies'
        ]
    }

    # Save lessons learned
    lessons_filename = f"lessons_learned_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced User Execution Detection',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "powershell.*-executionpolicy.*bypass"), 10, match(CommandLine, "cmd\.exe.*/c.*echo.*\|"), 9, match(CommandLine, "rundll32.*javascript:"), 10, 1==1, 4) | where risk_score >= 7',
            'alert_threshold': 5,
            'time_window': '15m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk correlation rules')
        print(f"   Updated Splunk correlation rules for user execution detection")

    # Update CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'user_execution_prevention': 'enabled',
        'script_execution_control': 'strict',
        'malicious_file_prevention': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike prevention policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule security awareness training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Safe Email and File Handling',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled security awareness training')
        print(f"   Scheduled security awareness training for affected users")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in user_execution_indicators:
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'Incident successfully contained, eradicated, and recovered',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(terminated_processes) > 0 or len(deleted_files) > 0,
                'systems_recovered': len(reenabled_hosts) > 0,
                'preventive_measures': len(preventive_measures) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Response effectiveness: {'HIGH' if len(isolated_hosts) > 0 and len(terminated_processes) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
